In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('play_tennis.csv')

In [5]:
df

,day,outlook,temp,humidity,wind,play
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes
5,D6,Rain,Cool,Normal,Strong,No
6,D7,Overcast,Cool,Normal,Strong,Yes
7,D8,Sunny,Mild,High,Weak,No
8,D9,Sunny,Cool,Normal,Weak,Yes
9,D10,Rain,Mild,Normal,Weak,Yes


In [6]:
df = df.drop(columns='day')

In [7]:
df

,outlook,temp,humidity,wind,play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


# Manual

In [11]:
df['play'].value_counts()

play
Yes    9
No     5
Name: count, dtype: int64

In [12]:
Pyes = 9/14
Pno = 5/14

In [14]:
pd.crosstab(df['outlook'], df['play'], margins=True)

play,No,Yes,All
outlook,,,
Overcast,0,4,4
Rain,2,3,5
Sunny,3,2,5
All,5,9,14


In [18]:
POvercastNo = 0/5
PRainyNo = 2/5
PSunnyNo = 3/5
POvercastYes = 4/9
PRainyYes = 3/9
PSunnyYes = 2/9

In [17]:
pd.crosstab(df['temp'], df['play'], margins=True)

play,No,Yes,All
temp,,,
Cool,1,3,4
Hot,2,2,4
Mild,2,4,6
All,5,9,14


In [19]:
PCoolNo = 1/5
PHotNo = 2/5
PMildNo = 2/5
PCoolYes = 3/9
PHotYes = 2/9
PMildYes = 4/9

In [20]:
pd.crosstab(df['humidity'], df['play'], margins=True)

play,No,Yes,All
humidity,,,
High,4,3,7
Normal,1,6,7
All,5,9,14


In [21]:
PHighNo = 4/5
PNormalNo = 1/5
PHighYes = 3/9
PNormalYes = 6/9

In [22]:
pd.crosstab(df['wind'], df['play'], margins=True)

play,No,Yes,All
wind,,,
Strong,3,3,6
Weak,2,6,8
All,5,9,14


In [23]:
PStrongNo = 3/5
PWeakNo = 2/5
PStrongYes = 3/9
PWeakYes = 6/9

### Prediction for `Overcast, Hot, Normal, Weak`

In [25]:
yes = Pyes*POvercastYes*PHotYes*PNormalYes*PWeakYes
no = Pno*POvercastNo*PHotNo*PNormalNo*PWeakNo

In [27]:
yes, no

(0.02821869488536155, 0.0)

In [26]:
print("Play" if yes>no else "No")

Play


# sklearn

In [38]:
X = df.drop(columns='play')
y = df['play']

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [39]:
labelencode = LabelEncoder()
y_encoded = labelencode.fit_transform(y)

In [41]:
categorical_col = ['outlook', 'temp', 'humidity', 'wind']

In [42]:
preprocessor = ColumnTransformer(transformers=[
    ('categorical', OrdinalEncoder(), categorical_col)
])

In [43]:
pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', CategoricalNB())
])

In [44]:
pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U3](2,)","['No','Yes']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['outlook','temp','humidity','wind']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified

In [46]:
from sklearn.metrics import accuracy_score
accuracy_score(y, pipeline.predict(X))

0.9285714285714286

### There are multiple algorithms out there for different type of data
BernoulliNB, CategoricalNB, ComplementNB, GaussianNB and MultinomialNB  
Each used for different type of features.

Naive bayes cant handle Mixed data (numeric, categorical and text).  
For it we learn RandomForest and XGBoost in future